In [1]:
import os, shutil, urllib.request, sys
from pathlib import Path
from pyflink.find_flink_home import _find_flink_home
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
import json

# Toggle this to True only for short debugging sessions.
DEBUG_STREAM_OUTPUT = False

# Fix Java on Windows.
if os.name == "nt":
    for home in [
        r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot",
        r"C:\Program Files\Adoptium\jdk-17.0.7",
    ]:
        java_bin = Path(home) / "bin" / "java.exe"
        if java_bin.exists():
            os.environ["JAVA_HOME"] = str(Path(home))
            os.environ["PATH"] = f"{java_bin.parent}{os.pathsep}" + os.environ.get("PATH", "")
            break

java_cmd = shutil.which("java")
if not java_cmd:
    raise RuntimeError("Java not found. Install JDK 11/17.")
print(f"Using Java: {java_cmd}")

# Required jars: Kafka + Cassandra connectors and Kafka client dependency.
flink_lib = Path(_find_flink_home()) / "lib"
jars = {
    "flink-connector-kafka-3.0.2-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-kafka/3.0.2-1.18/flink-connector-kafka-3.0.2-1.18.jar",
    "kafka-clients-3.2.3.jar":
        "https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.2.3/kafka-clients-3.2.3.jar",
    "flink-connector-cassandra_2.12-3.2.0-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-cassandra_2.12/3.2.0-1.18/flink-connector-cassandra_2.12-3.2.0-1.18.jar",
}

for jar_name, url in jars.items():
    dest = flink_lib / jar_name
    if not dest.exists():
        print(f"Downloading {jar_name} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"  -> Saved to {dest}")
    else:
        print(f"  OK: {jar_name} already present")

# Force Flink Python operators to use the same interpreter as this Jupyter kernel.
os.environ["PYFLINK_CLIENT_EXECUTABLE"] = sys.executable
os.environ["PYFLINK_PYTHON_EXECUTABLE"] = sys.executable

env = StreamExecutionEnvironment.get_execution_environment()
env.set_python_executable(sys.executable)
env.set_parallelism(1)
print(f"PyFlink worker python: {sys.executable}")
print("Flink environment initialized.")

Using Java: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
  OK: flink-connector-kafka-3.0.2-1.18.jar already present
  OK: kafka-clients-3.2.3.jar already present
  OK: flink-connector-cassandra_2.12-3.2.0-1.18.jar already present
PyFlink worker python: c:\Users\nmira\AppData\Local\Programs\Python\Python311\python.exe
Flink environment initialized.


In [2]:
# Cell 2: Source & Schema
kafka_props = {
    'bootstrap.servers': 'localhost:9094', #extrernal port mapped to kafka broker
    'group.id': 'taasim-jupyter-debugger', #grouping in consumer helps load balancing(fault tolerance and parallelism)
    'auto.offset.reset': 'earliest' #start consuming from earliest messages (all historical data)
}

kafka_source = FlinkKafkaConsumer(
    topics='raw.gps',
    deserialization_schema=SimpleStringSchema(), #will handle deserialization manually in next step
    properties=kafka_props
)
raw_stream = env.add_source(kafka_source) #creates data stream from kafka source

def parse_gps_json(data_string): 
    try:
        data = json.loads(data_string)
        return (
            int(data['taxi_id']),
            int(data['timestamp']),
            float(data['latitude']),
            float(data['longitude']),
            float(data['speed']),
            str(data['status'])
        )
    except Exception:
        return None
#structed stream with schema 
structured_stream = raw_stream \
    .map(
        parse_gps_json,
        output_type=Types.TUPLE([
            Types.INT(), Types.LONG(), Types.FLOAT(),
            Types.FLOAT(), Types.FLOAT(), Types.STRING()
        ])
    ) \
    .filter(lambda x: x is not None)

print("✅ Data source and schema mapped.")


✅ Data source and schema mapped.


In [3]:
# Cell 3: Watermark Strategy
from pyflink.common.watermark_strategy import WatermarkStrategy, TimestampAssigner
from pyflink.common.time import Duration

class GPSTimestampAssigner(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp):
       #extract GPS timestamp and convert it to milliseconds because fo flink requiremenets (millisceoncds)
        return int(value[1] * 1000)

watermark_strategy = WatermarkStrategy \
        .for_bounded_out_of_orderness(Duration.of_minutes(3)) \
        .with_timestamp_assigner(GPSTimestampAssigner())
#apply strategy to structed stream to get event time with watermarks
event_time_stream = structured_stream.assign_timestamps_and_watermarks(watermark_strategy)

if DEBUG_STREAM_OUTPUT:
    event_time_stream.print()
    print("Debug sink attached for event_time_stream.")
else:
    print("Debug sink disabled for event_time_stream.")

print("✅ Watermarks assigned.")

Debug sink disabled for event_time_stream.
✅ Watermarks assigned.


In [4]:
if "job_client" in globals():
    try:
        job_id = job_client.get_job_id()
        status = job_client.get_job_status().result()
        print(f"JobID: {job_id}")
        print(f"Job status: {status}")
        print("Job runs asynchronously in the notebook kernel.")
    except Exception as ex:
        print(f"Could not read job status: {ex}")
else:
    print("No job submitted yet. Run the final execution cell first.")

No job submitted yet. Run the final execution cell first.


In [5]:
def is_valid_gps(record):
    latitude = record[2]
    longitude = record[3]
    speed = record[4]

    if speed > 150.0:
        return False

    valid_lat = 33.4 <= latitude <= 33.7
    valid_lon = -7.8 <= longitude <= -7.4
    return valid_lat and valid_lon

valid_stream = event_time_stream.filter(is_valid_gps)
if DEBUG_STREAM_OUTPUT:
    valid_stream.print()
    print("Debug sink attached for valid_stream.")
else:
    print("Debug sink disabled for valid_stream.")

print("Validation filters configured: speed <= 150 and Casablanca bounding box.")

Debug sink disabled for valid_stream.
Validation filters configured: speed <= 150 and Casablanca bounding box.


In [6]:
CASABLANCA_ZONES = {
    1: {"name": "Anfa", "min_lat": 33.58, "max_lat": 33.62, "min_lon": -7.66, "max_lon": -7.62, "centroid_lat": 33.60, "centroid_lon": -7.64},
    2: {"name": "Sidi Belyout", "min_lat": 33.58, "max_lat": 33.61, "min_lon": -7.62, "max_lon": -7.58, "centroid_lat": 33.595, "centroid_lon": -7.60},
    3: {"name": "Roches Noires", "min_lat": 33.60, "max_lat": 33.64, "min_lon": -7.58, "max_lon": -7.54, "centroid_lat": 33.62, "centroid_lon": -7.56},
    4: {"name": "Ain Sebaa", "min_lat": 33.64, "max_lat": 33.70, "min_lon": -7.54, "max_lon": -7.48, "centroid_lat": 33.67, "centroid_lon": -7.51},
    5: {"name": "Hay Mohammadi", "min_lat": 33.60, "max_lat": 33.65, "min_lon": -7.56, "max_lon": -7.50, "centroid_lat": 33.625, "centroid_lon": -7.53},
    6: {"name": "Mers Sultan", "min_lat": 33.57, "max_lat": 33.60, "min_lon": -7.62, "max_lon": -7.58, "centroid_lat": 33.585, "centroid_lon": -7.60},
    7: {"name": "Maarif", "min_lat": 33.57, "max_lat": 33.60, "min_lon": -7.64, "max_lon": -7.60, "centroid_lat": 33.585, "centroid_lon": -7.62},
    8: {"name": "Sidi Othmane", "min_lat": 33.55, "max_lat": 33.59, "min_lon": -7.58, "max_lon": -7.52, "centroid_lat": 33.57, "centroid_lon": -7.55},
    9: {"name": "Sbata", "min_lat": 33.56, "max_lat": 33.59, "min_lon": -7.56, "max_lon": -7.50, "centroid_lat": 33.575, "centroid_lon": -7.53},
    10: {"name": "Ben M'Sick", "min_lat": 33.55, "max_lat": 33.58, "min_lon": -7.52, "max_lon": -7.46, "centroid_lat": 33.565, "centroid_lon": -7.49},
    11: {"name": "Ain Chock", "min_lat": 33.54, "max_lat": 33.58, "min_lon": -7.60, "max_lon": -7.54, "centroid_lat": 33.56, "centroid_lon": -7.57},
    12: {"name": "Hay Hassani", "min_lat": 33.54, "max_lat": 33.58, "min_lon": -7.66, "max_lon": -7.60, "centroid_lat": 33.56, "centroid_lon": -7.63},
    13: {"name": "Sidi Abderrahmane", "min_lat": 33.56, "max_lat": 33.60, "min_lon": -7.70, "max_lon": -7.66, "centroid_lat": 33.58, "centroid_lon": -7.68},
    14: {"name": "Ain Diab", "min_lat": 33.58, "max_lat": 33.62, "min_lon": -7.70, "max_lon": -7.66, "centroid_lat": 33.60, "centroid_lon": -7.68},
    15: {"name": "Dar Bouazza", "min_lat": 33.52, "max_lat": 33.58, "min_lon": -7.74, "max_lon": -7.66, "centroid_lat": 33.55, "centroid_lon": -7.70},
    16: {"name": "Sidi Moumen", "min_lat": 33.62, "max_lat": 33.68, "min_lon": -7.52, "max_lon": -7.46, "centroid_lat": 33.65, "centroid_lon": -7.49},
}

def map_and_anonymize(record):
    taxi_id = record[0]
    event_time = record[1]
    raw_lat = record[2]
    raw_lon = record[3]
    speed = record[4]
    status = record[5]

    city = "Casablanca"
    matched_zone_id = 99
    final_lat = raw_lat
    final_lon = raw_lon

    for zone_id, zone_data in CASABLANCA_ZONES.items():
        inside_lat = zone_data["min_lat"] <= raw_lat <= zone_data["max_lat"]
        inside_lon = zone_data["min_lon"] <= raw_lon <= zone_data["max_lon"]
        if inside_lat and inside_lon:
            matched_zone_id = zone_id
            final_lat = zone_data["centroid_lat"]
            final_lon = zone_data["centroid_lon"]
            break

    return (
        city,
        int(matched_zone_id),
        int(event_time),
        int(taxi_id),
        float(final_lat),
        float(final_lon),
        float(speed),
        str(status),
    )

anonymized_stream = valid_stream.map(
    map_and_anonymize,
    output_type=Types.TUPLE([
        Types.STRING(), Types.INT(), Types.LONG(), Types.INT(),
        Types.DOUBLE(), Types.DOUBLE(), Types.DOUBLE(), Types.STRING()
    ])
)

# Filter out zone_id 99 (out-of-scope GPS coordinates outside Casablanca zones)
filtered_anonymized_stream = anonymized_stream.filter(lambda record: record[1] != 99)

if DEBUG_STREAM_OUTPUT:
    filtered_anonymized_stream.print()
    print("Debug sink attached for filtered_anonymized_stream.")
else:
    print("Debug sink disabled for filtered_anonymized_stream.")

print("Map-matching, anonymization, and zone_id 99 filtering configured.")

Debug sink disabled for filtered_anonymized_stream.
Map-matching, anonymization, and zone_id 99 filtering configured.


In [7]:
from pyflink.datastream.connectors.kafka import FlinkKafkaProducer
import time

def tuple_to_json(record):
    payload = {
        "schema_version": "1.0.0",
        "event_type": "GPS_NORMALIZED",
        "event_time": int(record[2]),
        "ingested_at": int(time.time()),
        "taxi_id": int(record[3]),
        "zone_id": int(record[1]),
        "centroid_lat": float(record[4]),
        "centroid_lon": float(record[5]),
        "speed_kmh": float(record[6]),
        "status": str(record[7]),
    }
    return json.dumps(payload)

json_stream = filtered_anonymized_stream.map(tuple_to_json, output_type=Types.STRING())
kafka_producer_props = {
    "bootstrap.servers": "localhost:9094",
    "transaction.timeout.ms": "900000",
}

kafka_sink = FlinkKafkaProducer(
    topic="processed.gps",
    serialization_schema=SimpleStringSchema(),
    producer_config=kafka_producer_props,
)

json_stream.add_sink(kafka_sink)
print("Kafka sink configured for processed.gps (zone_id 99 filtered out).")

Kafka sink configured for processed.gps (zone_id 99 filtered out).


In [8]:
from pyflink.datastream.connectors.cassandra import CassandraSink

cassandra_query = (
    "INSERT INTO taasim.vehicle_positions "
    "(city, zone_id, event_time, taxi_id, latitude, longitude, speed, status) "
    "VALUES (?, ?, ?, ?, ?, ?, ?, ?);"
 )

cassandra_sink = (
    CassandraSink.add_sink(filtered_anonymized_stream)
    .set_host("localhost", 9042)
    .set_query(cassandra_query)
    .build()
 )

print("Cassandra sink configured for taasim.vehicle_positions (zone_id 99 filtered out).")
print("If running from containerized Flink, change host to cassandra.")

Cassandra sink configured for taasim.vehicle_positions (zone_id 99 filtered out).
If running from containerized Flink, change host to cassandra.


In [9]:
# Final execution cell: submit once after all sinks are configured.
should_submit = True

if "job_client" in globals():
    try:
        current_status = job_client.get_job_status().result()
        print(f"Existing job status: {current_status}")
        print(f"Existing JobID: {job_client.get_job_id()}")

        if "RUNNING" in str(current_status):
            print("Job is already running. Skipping re-submit to avoid graph reset errors.")
            should_submit = False
        else:
            print("Existing job is not running. Submitting a fresh job...")
    except Exception as ex:
        print(f"Existing job client is stale/unavailable: {ex}")
        print("Submitting a fresh job...")

if should_submit:
    job_client = env.execute_async("Jupyter: Job 1 - Full GPS Normalizer Pipeline")
    print(f"Job submitted. JobID: {job_client.get_job_id()}")

Job submitted. JobID: 520bcd387f65a95df34e8d57959ff550


---

## 📊 Job 2 — Demand Aggregator

**What this job does:** Every 30 seconds, per Casablanca zone, it counts how many taxis are active and how many trip requests are pending, then computes the supply/demand ratio.

**Data flow:**

processed.gps  ──┐
├──► union ──► key_by(zone_id) ──► 30s tumbling window ──► aggregate ──► Cassandra demand_zones
raw.trips      ──┘                                                                      └──► Kafka processed.demand

**Key concepts used:**
- **Event-time + watermarks (3 min):** same strategy as Job 1 — we trust the timestamp inside the event, not the system clock
- **`key_by(zone_id)`:** partitions the stream so each zone is processed independently
- **Tumbling window (30 s):** non-overlapping buckets; at the end of each bucket Flink emits one aggregate row per zone
- **`AggregateFunction`:** incrementally builds a set of unique vehicle IDs and trip IDs as events arrive — sets prevent double-counting
- **`WindowFunction`:** called once per closed window to attach the window's start timestamp to the result

In [10]:
import os, sys, json, time, urllib.request
from pathlib import Path

os.environ["_JAVA_OPTIONS"] = "-Djava.net.preferIPv4Stack=true"

from pyflink.find_flink_home import _find_flink_home
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer, FlinkKafkaProducer
from pyflink.datastream.connectors.cassandra import CassandraSink
from pyflink.datastream.functions import AggregateFunction, WindowFunction
from pyflink.datastream.window import TumblingEventTimeWindows
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
from pyflink.common.watermark_strategy import WatermarkStrategy, TimestampAssigner
from pyflink.common.time import Duration, Time

# ── Java path (Windows only) ──────────────────────────────────────────────────
if os.name == "nt":
    for home in [
        r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot",
        r"C:\Program Files\Adoptium\jdk-17.0.7",
        r"C:\Program Files\Java\jdk-17",
    ]:
        java_bin = Path(home) / "bin" / "java.exe"
        if java_bin.exists():
            os.environ["JAVA_HOME"] = str(Path(home))
            os.environ["PATH"] = f"{java_bin.parent}{os.pathsep}" + os.environ.get("PATH", "")
            break

# ── Ensure connector JARs are present ────────────────────────────────────────
flink_lib = Path(_find_flink_home()) / "lib"
JARS = {
    "flink-connector-kafka-3.0.2-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-kafka/3.0.2-1.18/flink-connector-kafka-3.0.2-1.18.jar",
    "kafka-clients-3.2.3.jar":
        "https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.2.3/kafka-clients-3.2.3.jar",
    "flink-connector-cassandra_2.12-3.2.0-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-cassandra_2.12/3.2.0-1.18/flink-connector-cassandra_2.12-3.2.0-1.18.jar",
}
for name, url in JARS.items():
    dest = flink_lib / name
    if not dest.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, dest)

os.environ["PYFLINK_CLIENT_EXECUTABLE"] = sys.executable
os.environ["PYFLINK_PYTHON_EXECUTABLE"] = sys.executable

# ── Flink environment ─────────────────────────────────────────────────────────
env = StreamExecutionEnvironment.get_execution_environment()
env.set_python_executable(sys.executable)
env.set_parallelism(1)

# ── Checkpointing (fault tolerance) ──────────────────────────────────────────
from pyflink.datastream import CheckpointingMode
env.enable_checkpointing(60_000, CheckpointingMode.EXACTLY_ONCE)
ckpt_path = os.path.abspath("cache/checkpoints/job2")
os.makedirs(ckpt_path, exist_ok=True)
env.get_checkpoint_config().set_checkpoint_storage_dir(
    f"file:///{ckpt_path.replace(os.sep, '/')}"
)

# ── Kafka connection properties ───────────────────────────────────────────────
KAFKA_PROPS = {
    "bootstrap.servers": "127.0.0.1:9094",
    "group.id": "taasim-job2-demand",
    "auto.offset.reset": "earliest",
}

# ── Read both input topics ────────────────────────────────────────────────────
gps_raw   = env.add_source(FlinkKafkaConsumer("processed.gps", SimpleStringSchema(), properties=KAFKA_PROPS))
trips_raw = env.add_source(FlinkKafkaConsumer("raw.trips",     SimpleStringSchema(), properties=KAFKA_PROPS))

# ── Parse helpers ─────────────────────────────────────────────────────────────
# Output schema: (event_type, zone_id, event_time_ms, entity_id)
# event_type = "vehicle" or "trip"  →  used inside the AggregateFunction
RECORD_TYPE = Types.TUPLE([Types.STRING(), Types.INT(), Types.LONG(), Types.STRING()])


"""Kafka delivers everything as raw text strings. These two functions 
convert them into a clean 4-field tuple that Flink can work with"""
def parse_gps(s):
    try:
        d = json.loads(s)
        return ("vehicle", int(d["zone_id"]), int(d["event_time"]) * 1000, str(d["taxi_id"]))
    except Exception:
        return None

def parse_trip(s):
    try:
        d = json.loads(s)
        return ("trip", int(d["origin_zone"]), int(d["requested_at"]), str(d["trip_id"]))
    except Exception:
        return None

gps_parsed   = gps_raw.map(parse_gps,   output_type=RECORD_TYPE).filter(lambda x: x is not None)
trips_parsed = trips_raw.map(parse_trip, output_type=RECORD_TYPE).filter(lambda x: x is not None)

# ── Assign event-time watermarks (3-minute allowed lateness) ─────────────────
class EventTimeExtractor(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp):
        return value[2]   # field index 2 = event_time_ms

watermark_strategy = (
    WatermarkStrategy
    .for_bounded_out_of_orderness(Duration.of_minutes(3)) #wait up to 3 minutes for stragglers before declaring a window closed.
    .with_timestamp_assigner(EventTimeExtractor()) #tell flink where to find event time in our tuple
)

#This merges both conveyor belts into one, flink does not care where event came from anymore, it just continuous flow of tuples tagged as vehicule or trip
combined = gps_parsed.union(trips_parsed).assign_timestamps_and_watermarks(watermark_strategy)

print("✅ Job 2 — sources and watermarks configured.")

✅ Job 2 — sources and watermarks configured.


In [ ]:

# ── Window logic ──────────────────────────────────────────────────────────────
# AggregateFunction: runs incrementally as events arrive inside the window.
# It keeps two sets (vehicle IDs, trip IDs) to count unique entities per zone.

#THE COUNTING ENGINE 
class DemandAggregateFunction(AggregateFunction):
    def create_accumulator(self): #when new 30Seconds window opens for a zone , to create fresh empty tally 
        return {"vehicles": set(), "trips": set(), "zone_id": None}

    def add(self, value, acc):
        event_type, zone_id, _, entity_id = value
        acc["zone_id"] = zone_id
        if event_type == "vehicle":
            acc["vehicles"].add(entity_id)
        else:
            acc["trips"].add(entity_id)
        return acc
    #it counts the set sizes and computes ratios
    def get_result(self, acc):
        v = len(acc["vehicles"])
        t = len(acc["trips"])
        return (acc["zone_id"], v, t, t / max(v, 1))   # (zone, vehicles, trips, ratio)

    def merge(self, a, b):
        a["vehicles"].update(b["vehicles"])
        a["trips"].update(b["trips"])
        return a


# WindowFunction: called once when the 30-second window closes.
# It attaches the window start timestamp to the aggregated result.

class DemandWindowFunction(WindowFunction):
    def apply(self, key, window, inputs):
        for result in inputs:
            zone_id, vehicles, trips, ratio = result
            yield (zone_id, vehicles, trips, ratio, int(window.start))
            break   # AggregateFunction already merged everything; only one result


# ── Key by zone, apply 30-second tumbling window ──────────────────────────────
WINDOW_OUTPUT_TYPE = Types.TUPLE([Types.INT(), Types.INT(), Types.INT(), Types.DOUBLE(), Types.LONG()])

aggregated = (
    combined
    .key_by(lambda r: r[1])                                  # r[1] = zone_id
    .window(TumblingEventTimeWindows.of(Time.seconds(30)))
    .aggregate(DemandAggregateFunction(), window_function=DemandWindowFunction(),
               output_type=WINDOW_OUTPUT_TYPE)
)
# aggregated schema: (zone_id, active_vehicles, pending_requests, ratio, window_start_ms)

# ── Cassandra sink ────────────────────────────────────────────────────────────
# Schema: (city, zone_id, window_start, active_vehicles, pending_requests, ratio)
cassandra_stream = aggregated.map(
    lambda r: ("Casablanca", int(r[0]), int(r[4]), int(r[1]), int(r[2]), float(r[3])),
    output_type=Types.TUPLE([Types.STRING(), Types.INT(), Types.LONG(), Types.INT(), Types.INT(), Types.DOUBLE()])
)

CassandraSink.add_sink(cassandra_stream) \
    .set_host("127.0.0.1", 9042) \
    .set_query("INSERT INTO taasim.demand_zones (city, zone_id, window_start, active_vehicles, pending_requests, ratio) VALUES (?, ?, ?, ?, ?, ?);") \
    .build()

# ── Kafka sink (processed.demand) ────────────────────────────────────────────
def to_demand_json(r):
    return json.dumps({
        "event_type": "DEMAND_AGGREGATED", "city": "Casablanca",
        "zone_id": int(r[0]), "active_vehicles": int(r[1]),
        "pending_requests": int(r[2]), "supply_demand_ratio": round(float(r[3]), 3),
        "window_start_ms": int(r[4]), "ingested_at": int(time.time() * 1000),
    })

aggregated.map(to_demand_json, output_type=Types.STRING()) \
    .add_sink(FlinkKafkaProducer(
        "processed.demand", SimpleStringSchema(),
        producer_config={"bootstrap.servers": "127.0.0.1:9094", "transaction.timeout.ms": "900000"}
    ))

print("✅ Job 2 — sinks configured. Submitting...")
env.execute("Standalone: Job 2 - Demand Aggregation")

✅ Job 2 — sinks configured. Submitting...


---

## 🚖 Job 3 — Trip Matcher

**What this job does:** For every incoming trip request, find the nearest available taxi in the same zone and compute an ETA. If no taxi is found within 5 seconds, expand the search to adjacent zones.

**Data flow:**

processed.gps ──┐
├──► union ──► key_by("Casablanca") ──► KeyedProcessFunction ──► Cassandra trips
raw.trips     ──┘                                                              └──► Kafka processed.trips

**Key concepts used:**
- **`KeyedProcessFunction`:** unlike windowed jobs, this function reacts to *every single event* and holds persistent state between events
- **`MapState` (Flink managed state):** two state maps survive crashes thanks to checkpointing
  - `vehicles_state` — latest known location of every taxi (updated on every GPS ping)
  - `pending_trips` — trip requests waiting for a taxi (populated when no immediate match is found)
- **Processing-time timers:** when a trip has no immediate match, we register a 5-second timer; when it fires we do a fallback search in adjacent zones
- **Haversine ETA:** straight-line distance between the rider's origin and the matched taxi's zone centroid, divided by the taxi's reported speed

**Matching priority:**
1. Immediate match in the **same zone** → `MATCHED_EXACT`
2. After 5 s timer → retry same zone → `MATCHED_DELAYED_EXACT`
3. After 5 s timer → expand to **adjacent zones** → `MATCHED_FALLBACK` (ETA +5 min)
4. No taxi anywhere → `UNFULFILLED`

In [ ]:
import os, sys, json, time, math, urllib.request
from pathlib import Path

os.environ["_JAVA_OPTIONS"] = "-Djava.net.preferIPv4Stack=true"

from pyflink.find_flink_home import _find_flink_home
from pyflink.datastream import StreamExecutionEnvironment, CheckpointingMode
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer, FlinkKafkaProducer
from pyflink.datastream.connectors.cassandra import CassandraSink
from pyflink.datastream.functions import KeyedProcessFunction
from pyflink.datastream.state import MapStateDescriptor
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types

# ── Java path (Windows only) ──────────────────────────────────────────────────
if os.name == "nt":
    for home in [
        r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot",
        r"C:\Program Files\Adoptium\jdk-17.0.7",
        r"C:\Program Files\Java\jdk-17",
    ]:
        java_bin = Path(home) / "bin" / "java.exe"
        if java_bin.exists():
            os.environ["JAVA_HOME"] = str(Path(home))
            os.environ["PATH"] = f"{java_bin.parent}{os.pathsep}" + os.environ.get("PATH", "")
            break

# ── Ensure connector JARs are present ────────────────────────────────────────
flink_lib = Path(_find_flink_home()) / "lib"
JARS = {
    "flink-connector-kafka-3.0.2-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-kafka/3.0.2-1.18/flink-connector-kafka-3.0.2-1.18.jar",
    "kafka-clients-3.2.3.jar":
        "https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.2.3/kafka-clients-3.2.3.jar",
    "flink-connector-cassandra_2.12-3.2.0-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-cassandra_2.12/3.2.0-1.18/flink-connector-cassandra_2.12-3.2.0-1.18.jar",
}
for name, url in JARS.items():
    dest = flink_lib / name
    if not dest.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, dest)

os.environ["PYFLINK_CLIENT_EXECUTABLE"] = sys.executable
os.environ["PYFLINK_PYTHON_EXECUTABLE"] = sys.executable

# ── Flink environment ─────────────────────────────────────────────────────────
env = StreamExecutionEnvironment.get_execution_environment()
env.set_python_executable(sys.executable)
env.set_parallelism(1)

# ── Checkpointing ─────────────────────────────────────────────────────────────
env.enable_checkpointing(60_000, CheckpointingMode.EXACTLY_ONCE)
ckpt_path = os.path.abspath("cache/checkpoints/job3")
os.makedirs(ckpt_path, exist_ok=True)
env.get_checkpoint_config().set_checkpoint_storage_dir(
    f"file:///{ckpt_path.replace(os.sep, '/')}"
)

# ── Kafka sources ─────────────────────────────────────────────────────────────
KAFKA_PROPS = {
    "bootstrap.servers": "127.0.0.1:9094",
    "group.id": "taasim-job3-matcher",
    "auto.offset.reset": "earliest",
}

gps_raw   = env.add_source(FlinkKafkaConsumer("processed.gps", SimpleStringSchema(), properties=KAFKA_PROPS))
trips_raw = env.add_source(FlinkKafkaConsumer("raw.trips",     SimpleStringSchema(), properties=KAFKA_PROPS))

# ── Tag each event with its source, key everything by city so it flows to one node ──
# Output schema: (source_type, city_key, raw_json_string)
TAGGED_TYPE = Types.TUPLE([Types.STRING(), Types.STRING(), Types.STRING()])

def tag(src):
    def _tag(s):
        try:
            json.loads(s)          # validate JSON before forwarding
            return (src, "Casablanca", s)
        except Exception:
            return None
    return _tag

gps_tagged   = gps_raw.map(tag("gps"),  output_type=TAGGED_TYPE).filter(lambda x: x is not None)
trips_tagged = trips_raw.map(tag("trip"), output_type=TAGGED_TYPE).filter(lambda x: x is not None)

# Union and key by city → all events go to the same KeyedProcessFunction instance
keyed = gps_tagged.union(trips_tagged).key_by(lambda r: r[1])

print("✅ Job 3 — sources configured.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Job 3 │ Step 2 — TripMatcherFunction → Cassandra + Kafka sinks → Execute
# ─────────────────────────────────────────────────────────────────────────────
#compute sthe great circle distance (shortest)
def haversine_km(lat1, lon1, lat2, lon2):
    """Straight-line distance between two GPS points (km)."""
    R = 6371
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


#this functions runs on every single event and has access to persistent state survives between events and crashes
class TripMatcherFunction(KeyedProcessFunction):
    """
    Stateful function that holds a map of available vehicles and a map of
    pending (unmatched) trips.  On each event:
      - GPS ping  → update vehicles_state
      - Trip req  → try immediate match; if none, park in pending_trips + set 5s timer
      - Timer     → retry; fallback to adjacent zones; emit UNFULFILLED if still nothing
    """

    def open(self, ctx):
        self.vehicles_state = ctx.get_map_state(
            MapStateDescriptor("vehicles", Types.STRING(), Types.STRING())
        )
        self.pending_trips = ctx.get_map_state(
            MapStateDescriptor("pending_trips", Types.STRING(), Types.STRING())
        )

    # ── helpers ───────────────────────────────────────────────────────────────
    #scans the viehicules states to find the first available taxi in requested zone
    def _find_vehicle(self, zone_id, origin_lat=None, origin_lon=None):
        """Return (taxi_id, eta_seconds) for the first available taxi in zone_id, or (None, 0)."""
        now_ms = int(time.time() * 1000)
        for v_id, v_json in self.vehicles_state.items():
            v = json.loads(v_json)
            age_ms  = now_ms - v.get("last_updated", 0)
            is_available = v.get("status", "") in ("AVAILABLE", "EN_ROUTE")
            in_zone  = int(v.get("zone_id", -1)) == int(zone_id)
            is_fresh = age_ms < 120_000      # GPS ping must be < 2 minutes old

            if is_available and in_zone and is_fresh:
                speed = max(float(v.get("speed", 30.0)), 10.0)
                if origin_lat is not None:
                    dist = haversine_km(origin_lat, origin_lon,
                                        float(v.get("lat", 0)), float(v.get("lon", 0)))
                    eta = max(int(dist / speed * 3600), 60)
                else:
                    eta = 180    # default 3-minute ETA when coordinates are missing
                return v_id, eta
        return None, 0
    
    # packaging the result called by on timer exatct match or fallback  process elemnt 
    def _emit_match(self, trip_id, trip, taxi_id, eta, status):
        return json.dumps({
            "city": "Casablanca",
            "date_bucket": time.strftime("%Y-%m-%d"),
            "created_at": trip.get("requested_at", int(time.time() * 1000)),
            "trip_id": trip_id,
            "rider_id": trip.get("rider_id", 0),
            "origin_zone": trip.get("origin_zone", 0),
            "destination_zone": trip.get("destination_zone", 0),
            "call_type": trip.get("call_type", "A"),
            "matched_taxi": taxi_id,
            "eta_seconds": eta,
            "status": status,
        })

    # ── main processing ───────────────────────────────────────────────────────
    #reacting to each event
    def process_element(self, value, ctx):
        src, _, payload_str = value
        payload = json.loads(payload_str)
       #gps event arrives we just overwrite taxi entry in vehicules state with fresh coordinates
        if src == "gps":
            # Update the vehicle's last known position in state
            v_id = str(payload.get("taxi_id"))
            self.vehicles_state.put(v_id, json.dumps({
                "zone_id":      payload.get("zone_id", 0),
                "lat":          payload.get("centroid_lat", payload.get("latitude", 0.0)),
                "lon":          payload.get("centroid_lon", payload.get("longitude", 0.0)),
                "speed":        payload.get("speed_kmh", payload.get("speed", 30.0)),
                "status":       payload.get("status", "AVAILABLE"),
                "last_updated": int(time.time() * 1000),
            }))
        #trip arrives we immidieatly call the find vehicule function
        elif src == "trip":
            trip_id    = str(payload.get("trip_id"))
            origin     = int(payload.get("origin_zone", 0))
            o_lat, o_lon = payload.get("origin_lat"), payload.get("origin_lon")

            taxi, eta = self._find_vehicle(origin, o_lat, o_lon)
            if taxi:
                self.vehicles_state.remove(taxi)
                yield self._emit_match(trip_id, payload, taxi, eta, "MATCHED_EXACT")
            else:
                # Park the trip and schedule a 5-second fallback attempt
                self.pending_trips.put(trip_id, payload_str)
                ctx.timer_service().register_processing_time_timer(
                    ctx.timer_service().current_processing_time() + 5_000
                )
    
    #the 5-second fallback alarm, 5sec after trip was parked , flink calls on timer , loops through every pending trip
    #retry same zone , adjacent zones, giveUp
    def on_timer(self, timestamp, ctx):
        """Fires 5 seconds after a trip was parked. Retries exact zone, then adjacent zones."""
        resolved = []

        for trip_id, trip_str in self.pending_trips.items():
            trip   = json.loads(trip_str)
            origin = int(trip.get("origin_zone", 0))
            o_lat, o_lon = trip.get("origin_lat"), trip.get("origin_lon")

            # 1. Retry exact zone
            taxi, eta = self._find_vehicle(origin, o_lat, o_lon)
            if taxi:
                self.vehicles_state.remove(taxi)
                yield self._emit_match(trip_id, trip, taxi, eta, "MATCHED_DELAYED_EXACT")
                resolved.append(trip_id)
                continue

            # 2. Expand to adjacent zones (neighbours = zone ± 1)
            for adj in (origin - 1, origin + 1):
                taxi, eta = self._find_vehicle(adj, o_lat, o_lon)
                if taxi:
                    self.vehicles_state.remove(taxi)
                    yield self._emit_match(trip_id, trip, taxi, eta + 300, "MATCHED_FALLBACK")
                    resolved.append(trip_id)
                    break
            else:
                # 3. No taxi found anywhere → mark unfulfilled
                yield self._emit_match(trip_id, trip, "NONE", 0, "UNFULFILLED")
                resolved.append(trip_id)

        for trip_id in resolved:
            self.pending_trips.remove(trip_id)


# ── Apply the function ────────────────────────────────────────────────────────
matched_json = keyed.process(TripMatcherFunction(), output_type=Types.STRING())

# ── Cassandra sink ────────────────────────────────────────────────────────────
def to_cassandra_tuple(s):
    r = json.loads(s)
    return (
        str(r["city"]), str(r["date_bucket"]), int(r["created_at"]),
        str(r["trip_id"]), int(r["rider_id"]),
        int(r["origin_zone"]), int(r["destination_zone"]),
        str(r["call_type"]), str(r["matched_taxi"]),
        int(r["eta_seconds"]), str(r["status"]),
    )

cassandra_stream = matched_json.map(
    to_cassandra_tuple,
    output_type=Types.TUPLE([
        Types.STRING(), Types.STRING(), Types.LONG(), Types.STRING(), Types.INT(),
        Types.INT(), Types.INT(), Types.STRING(), Types.STRING(), Types.INT(), Types.STRING(),
    ])
)

CassandraSink.add_sink(cassandra_stream) \
    .set_host("127.0.0.1", 9042) \
    .set_query(
        "INSERT INTO taasim.trips "
        "(city, date_bucket, created_at, trip_id, rider_id, "
        "origin_zone, destination_zone, call_type, matched_taxi, eta_seconds, status) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"
    ).build()

# ── Kafka sink (processed.trips) ──────────────────────────────────────────────
matched_json.add_sink(FlinkKafkaProducer(
    "processed.trips", SimpleStringSchema(),
    producer_config={"bootstrap.servers": "127.0.0.1:9094", "transaction.timeout.ms": "900000"}
))

matched_json.print()   # local debug output

print("✅ Job 3 — sinks configured. Submitting...")
env.execute("Standalone: Job 3 - Trip Matching")